# Creating the Following Features
- Total days with a text (Amount of days where two people texted at all)
- Total calls between the two
- Total days with a call (Amount of days where two people called at all)

## Total days each pair texted

Each pair is treated as unordered, so messages from user A to user B and from user B to user A belong to the same pair. Multiple messages exchanged by the pair during the same elapsed study day count as one texting day. No imputation or scaling is applied.

In [1]:
from pathlib import Path

import pandas as pd


SMS_INPUT_FILE = Path('../data/raw/sms.csv')
SMS_PAIR_DAYS_OUTPUT_FILE = Path('../data/interim/sms_pair_days.csv')
SECONDS_PER_DAY = 86_400
PAIR_COLUMNS = ['user_a', 'user_b']

sms_data = pd.read_csv(SMS_INPUT_FILE)
sms_data.head()

,timestamp,sender,recipient
0,18,370,512
1,37,512,370
2,126,370,512
3,134,0,512
4,312,137,136


In [2]:
# Fail instead of imputing if a required raw value is missing.
required_columns = ['timestamp', 'sender', 'recipient']
if sms_data[required_columns].isna().any().any():
    raise ValueError('SMS input contains missing values; no imputation was applied.')

# Canonicalize sender/recipient into one unordered pair.
sms_pair_events = sms_data[required_columns].copy()
sms_pair_events['user_a'] = sms_pair_events[['sender', 'recipient']].min(axis=1)
sms_pair_events['user_b'] = sms_pair_events[['sender', 'recipient']].max(axis=1)

# Convert elapsed seconds into elapsed study-day buckets. This is only used
# to identify distinct days; the resulting count is not scaled.
sms_pair_events['texting_day'] = (
    sms_pair_events['timestamp'] // SECONDS_PER_DAY
).astype('int64')

sms_pair_days = (
    sms_pair_events.loc[
        sms_pair_events['user_a'].lt(sms_pair_events['user_b']),
        PAIR_COLUMNS + ['texting_day'],
    ]
    .drop_duplicates()
    .groupby(PAIR_COLUMNS, as_index=False)
    .agg(days_texted=('texting_day', 'nunique'))
    .sort_values(PAIR_COLUMNS, ignore_index=True)
)

# Structural checks only: no imputation or scaling is performed.
assert not sms_pair_days[PAIR_COLUMNS].duplicated().any()
assert sms_pair_days['days_texted'].ge(1).all()

SMS_PAIR_DAYS_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
sms_pair_days.to_csv(SMS_PAIR_DAYS_OUTPUT_FILE, index=False)

print(f'Saved {len(sms_pair_days):,} SMS pairs to {SMS_PAIR_DAYS_OUTPUT_FILE}')
display(sms_pair_days.head())

Saved 697 SMS pairs to ..\data\interim\sms_pair_days.csv


,user_a,user_b,days_texted
0,0,208,1
1,0,512,7
2,1,345,1
3,3,49,14
4,3,217,2


## Total days each pair called

Each pair is treated as unordered, so calls in either direction belong to the same pair. Multiple calls during the same elapsed study day count as one calling day. Missed calls are included because they still indicate that the pair called that day. No imputation or scaling is applied.

In [3]:
CALLS_INPUT_FILE = Path('../data/raw/calls.csv')
CALL_PAIR_DAYS_OUTPUT_FILE = Path('../data/interim/call_pair_days.csv')

calls_data = pd.read_csv(CALLS_INPUT_FILE)
calls_data.head()

,timestamp,caller,callee,duration
0,184,300,301,121
1,3920,512,299,670
2,5623,301,300,504
3,9252,401,457,-1
4,15466,512,0,5


In [4]:
# Fail instead of imputing if a required raw value is missing.
required_call_columns = ['timestamp', 'caller', 'callee']
if calls_data[required_call_columns].isna().any().any():
    raise ValueError('Call input contains missing values; no imputation was applied.')

# Canonicalize caller/callee into one unordered pair.
call_pair_events = calls_data[required_call_columns].copy()
call_pair_events['user_a'] = call_pair_events[['caller', 'callee']].min(axis=1)
call_pair_events['user_b'] = call_pair_events[['caller', 'callee']].max(axis=1)

# Assign each call to an elapsed study-day bucket. This is only used to
# identify distinct days; the resulting count is not scaled.
call_pair_events['calling_day'] = (
    call_pair_events['timestamp'] // SECONDS_PER_DAY
).astype('int64')

call_pair_days = (
    call_pair_events.loc[
        call_pair_events['user_a'].lt(call_pair_events['user_b']),
        PAIR_COLUMNS + ['calling_day'],
    ]
    .drop_duplicates()
    .groupby(PAIR_COLUMNS, as_index=False)
    .agg(days_called=('calling_day', 'nunique'))
    .sort_values(PAIR_COLUMNS, ignore_index=True)
)

# Structural checks only: no imputation or scaling is performed.
assert not call_pair_days[PAIR_COLUMNS].duplicated().any()
assert call_pair_days['days_called'].ge(1).all()

CALL_PAIR_DAYS_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
call_pair_days.to_csv(CALL_PAIR_DAYS_OUTPUT_FILE, index=False)

print(f'Saved {len(call_pair_days):,} call pairs to {CALL_PAIR_DAYS_OUTPUT_FILE}')
display(call_pair_days.head())

Saved 621 call pairs to ..\data\interim\call_pair_days.csv


,user_a,user_b,days_called
0,0,208,2
1,0,512,4
2,3,48,1
3,3,49,5
4,3,357,6


## Total calls between each pair

Every call record counts once, regardless of the day it occurred. Calls in either direction belong to the same unordered pair, so two calls on one day and three calls on another day produce `total_calls = 5`. Missed calls are included as call attempts. No imputation or scaling is applied.

In [5]:
CALLS_INPUT_FILE = Path("../data/raw/calls.csv")
CALL_PAIR_TOTALS_OUTPUT_FILE = Path("../data/interim/call_pair_totals.csv")
PAIR_COLUMNS = ["user_a", "user_b"]

calls_data = pd.read_csv(CALLS_INPUT_FILE)

calls_data.head()

,timestamp,caller,callee,duration
0,184,300,301,121
1,3920,512,299,670
2,5623,301,300,504
3,9252,401,457,-1
4,15466,512,0,5


In [6]:
required_columns = ["caller", "callee"]

# Stop if required values are missing rather than applying imputation.
if calls_data[required_columns].isna().any().any():
    raise ValueError(
        "Call data contains missing caller or callee values; "
        "no imputation was applied."
    )

call_pair_events = calls_data[required_columns].copy()

# Create unordered pairs so A calling B and B calling A count together.
call_pair_events["user_a"] = call_pair_events[
    ["caller", "callee"]
].min(axis=1)

call_pair_events["user_b"] = call_pair_events[
    ["caller", "callee"]
].max(axis=1)

# Count every call record regardless of when it occurred.
call_pair_totals = (
    call_pair_events.loc[
        call_pair_events["user_a"] < call_pair_events["user_b"],
        PAIR_COLUMNS,
    ]
    .groupby(PAIR_COLUMNS, as_index=False)
    .size()
    .rename(columns={"size": "total_calls"})
    .sort_values(PAIR_COLUMNS, ignore_index=True)
)

# Validate the table without imputing or scaling.
assert not call_pair_totals[PAIR_COLUMNS].duplicated().any()
assert call_pair_totals["total_calls"].ge(1).all()

valid_call_count = (
    call_pair_events["user_a"] < call_pair_events["user_b"]
).sum()

assert call_pair_totals["total_calls"].sum() == valid_call_count

# Save and display the completed feature table.
CALL_PAIR_TOTALS_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
call_pair_totals.to_csv(CALL_PAIR_TOTALS_OUTPUT_FILE, index=False)

print(
    f"Saved {len(call_pair_totals):,} pairs and "
    f"{call_pair_totals['total_calls'].sum():,} total calls."
)

display(call_pair_totals)

Saved 621 pairs and 3,600 total calls.


,user_a,user_b,total_calls
0,0,208,4
1,0,512,5
2,3,48,2
3,3,49,6
4,3,357,8
...,...,...,...
616,673,674,1
617,677,681,1
618,677,700,2
619,692,702,14
